# 🔥 Smoke da forja — o loop de evolução roda ponta a ponta? (Degrau 2, sub-issue #5)

Notebook de smoke da **forja de políticas-código** (épica #1): 30 iterações do loop a partir da semente (`forge/seed_policy.py`, geração 0 = v0), no protocolo **pré-registrado**:

- **O loop:** pai → mutação → **guarda AST anti-trapaça** (`mundo.program.validate_program`) → **avaliador stage1** (8 episódios, campo F0, seeds de TREINO) → **stage2** (64 episódios de validação, F0+F1) quando S1 ≥ 0,45 (limiar da cascata do `forge/config.yaml`) ou S1 é recorde do loop. A iteração 0 é a semente, avaliada nos dois estágios. Seeds de TESTE (9901–9932) nunca entram — o avaliador trava isso.
- **O mutador deste smoke é PROGRAMÁTICO (AST), não LLM** — decisão registrada em `forge/mutator.py`: não há chave/endpoint de LLM neste ambiente e nenhum segredo novo entra sem o fundador. O smoke prova o *pipeline* (guarda, cascata, artefatos, curva de S); as mutações perturbam genes via `self.p[...]` (mesmos limites do torneio) e literais do EVOLVE-BLOCK. Qual LLM entra na corrida forge-v1 (#6) é decisão do fundador.
- **Portão:** as 30 iterações completam sem erro estrutural do driver, com registro por iteração (hash, pai, mutação, status, S1/S2, melhor S) → **SMOKE FORJA: OK** + o melhor S. Quebra de loop/guarda/driver → **SMOKE FORJA: FALHOU** + a causa.
- Baseline a bater (fora deste smoke): genoma campeão do torneio, S=0,563. Aqui a referência é a própria semente nos mesmos blocos.

Regra do fundador: experimento roda AQUI, no Kaggle — nunca na VM de desenvolvimento.


In [ ]:
# 1) Fonte do pacote: dataset privado do bundle (mundo-autonomo-src11).
#    LOADER RECURSIVO: o Kaggle expoe os datasets em
#    /kaggle/input/datasets/<dono>/<slug>/ com a arvore ja extraida;
#    a busca aceita qualquer 'src/mundo/__init__.py' sob /kaggle/input,
#    com fallback para o zip mundo-src.zip. O bundle do smoke leva
#    src/ + forge/ + pyproject.toml — a raiz do bundle entra no
#    sys.path porque a forja vive em forge/, fora do pacote.
import glob, os, sys, zipfile

inits = sorted(glob.glob('/kaggle/input/**/src/mundo/__init__.py',
                         recursive=True))
zip_cands = sorted(glob.glob('/kaggle/input/**/mundo*.zip', recursive=True))
if inits:
    bundle_root = os.path.dirname(os.path.dirname(
        os.path.dirname(inits[-1])))
elif zip_cands:
    zipfile.ZipFile(zip_cands[-1]).extractall('/kaggle/working/pkg')
    bundle_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada (loader recursivo): '
                     'anexe o dataset pascalcoimbra/mundo-autonomo-src11 '
                     'a este kernel.')
print('bundle em', bundle_root)
sys.path.insert(0, os.path.join(bundle_root, 'src'))
sys.path.insert(0, bundle_root)
import mundo
print('mundo', mundo.__version__, 'importado de', bundle_root)
for rel in ('forge/smoke.py', 'forge/mutator.py', 'forge/evaluator.py',
            'forge/seed_policy.py', 'src/mundo/program.py'):
    assert os.path.exists(os.path.join(bundle_root, rel)),         f'{rel} ausente no bundle'
print('forge/ completa no bundle')


In [ ]:
# 2) O smoke: 30 iteracoes do loop [mutacao -> guarda -> stage1
#    -> stage2 nos melhores], com registro por iteracao no stdout
#    e em /kaggle/working/smoke-forja/smoke_log.jsonl.
import importlib.util, json, os, time

spec = importlib.util.spec_from_file_location(
    'forge_smoke', os.path.join(bundle_root, 'forge', 'smoke.py'))
smoke = importlib.util.module_from_spec(spec)
spec.loader.exec_module(smoke)

t0 = time.time()
summary = smoke.run_smoke(iterations=30, seed=42,
                          out_dir='/kaggle/working/smoke-forja')
print(f'smoke concluido em {time.time() - t0:.0f}s')
print(json.dumps({k: v for k, v in summary.items() if k != 'records'},
                 ensure_ascii=False, indent=1))


In [ ]:
# 3) VEREDITO (regra pre-registrada no topo)
import json, os, shutil

ok = bool(summary.get('ok'))
best = summary.get('best_by_S2') or summary.get('best_by_S1') or {}
seed_res = summary.get('seed_result') or {}
print('=' * 64)
if ok:
    print('SMOKE FORJA: OK')
else:
    print('SMOKE FORJA: FALHOU')
    print('causa:', summary.get('fatal')
          or f"iteracoes completas: "
             f"{summary.get('iterations_completed')}/30")
print(f"iteracoes: {summary.get('iterations_completed')}/30 | "
      f"contagens: {summary.get('counts')}")
print(f"semente prog:{seed_res.get('hash')}: S1={seed_res.get('S1')} "
      f"S2={seed_res.get('S2')}")
print(f"melhor programa prog:{best.get('hash')}: S1={best.get('S1')} "
      f"S2={best.get('S2')}")
print(f"curva best_S1: {summary.get('s_curve_best_S1')}")
print('=' * 64)
with open('/kaggle/working/smoke-forja.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=1)
_best = '/kaggle/working/smoke-forja/best_program.py'
if os.path.exists(_best):
    shutil.copy(_best, '/kaggle/working/best_program.py')
